# Veil · Colab runner

Runs the full Veil pipeline on a free Colab GPU with **SAM 3.1** (Meta's open weights) in this notebook, then freezes the results for instant replay in the app:

`demo_assets/cache/<run>/` → `redacted.mp4`, `masks.mp4`, `audit.json`, `report.html` (tamper-evident Redaction Report), `comparison_grid.mp4`, `comparison_pair.mp4`

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (any NVIDIA GPU works; Ampere or newer runs native bf16).
2. Approved access to [facebook/sam3.1](https://huggingface.co/facebook/sam3.1), and a Hugging Face read token saved as a Colab secret named **`HF_TOKEN`** (key icon in the left sidebar; enable *Notebook access*).
3. Run the cells top to bottom. The smoke test (step 6) tells you how long the batch will take.

Nothing here needs a credit card. Colab is used as a batch job only; nothing is served from it.

In [ ]:
# 1 · GPU check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
major, minor = torch.cuda.get_device_capability()
print(f"torch {torch.__version__} · {torch.cuda.get_device_name()} · compute {major}.{minor} · native bf16: {major >= 8}")

In [ ]:
# 2 · Settings
REPO_URL = "https://github.com/atharva283/veil-meta-ai.git"
BRANCH = "hackathon-plan"

PRECISION = "auto"      # auto = bf16 on Ampere+, fp16 on T4. If the smoke test fails or finds nothing, use "fp32".
# A run takes ~10-55 min on a T4 (each concept is a full SAM pass, plus the audit's re-check),
# more than one free session holds in total, so build in batches and download after each (step 9):
#   batch 1: ["manchester_street", "desk_inbox"]   batch 2: ["kl_traffic"]   batch 3: ["storefront_crossing"]
CLIPS = ["manchester_street", "desk_inbox"]   # None = every real sample clip
MAX_PROMPTS = 1         # suggested instructions per clip (each costs one full run)
FOLLOW_UPS = True       # also run each clip's follow-up instruction
USE_LLAMA_PLANNER = True  # Llama 3.1 8B (Meta, via Ollama) plans the redaction; False = rule-based planner
SAVE_TO_DRIVE = True    # also copy the cache to Google Drive (MyDrive/veil_cache)

In [ ]:
# 3 · Code and dependencies (~2 min)
import os, subprocess
if os.path.exists("/content/veil"):
    !git -C /content/veil fetch -q --depth 1 origin {BRANCH} && git -C /content/veil checkout -q FETCH_HEAD
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/veil
%cd /content/veil/backend
!git log -1 --format="Veil @ %h (%cd)"

SAM_COMMIT = subprocess.check_output(
    ["python", "-c", "from app.segmenter.sam3_core import SAM_COMMIT; print(SAM_COMMIT)"], text=True).strip()
!pip install -q -r requirements.txt
# sam3 pins numpy<2; Colab ships numpy 2, which sam3's code runs fine on, so install it without its pins.
!pip install -q --no-deps "git+https://github.com/facebookresearch/sam3.git@{SAM_COMMIT}"
!pip install -q timm "ftfy==6.1.1" regex iopath einops scipy psutil huggingface_hub "setuptools<81"  # sam3 imports pkg_resources
!python -c "import sam3, torch, numpy; print('sam3 OK · torch', torch.__version__, '· numpy', numpy.__version__)"

In [ ]:
# 4 · Hugging Face access and SAM 3.1 weights (3.5 GB, ~1-2 min)
from google.colab import userdata
from huggingface_hub import hf_hub_download
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
for name in ("config.json", "sam3.1_multiplex.pt"):
    print(hf_hub_download("facebook/sam3.1", name))

In [ ]:
# 5 · Optional: Llama 3.1 8B planner via Ollama (~3 min)
# Ollama unloads the model after each call (KEEP_ALIVE=0) so SAM keeps the GPU memory.
if USE_LLAMA_PLANNER:
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_KEEP_ALIVE": "0"},
                     stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    import time; time.sleep(5)
    !ollama pull llama3.1:8b > /dev/null && echo "llama3.1:8b ready"
    os.environ.update(META_MODEL_API_BASE="http://localhost:11434/v1", META_MODEL_API_KEY="ollama", PLANNER_MODEL="llama3.1:8b")
else:
    for k in ("META_MODEL_API_BASE", "META_MODEL_API_KEY", "PLANNER_MODEL"):
        os.environ.pop(k, None)

In [ ]:
# 6 · Smoke test: SAM 3.1 on 240 frames (8 s). Runs in a fresh process, like the batch.
# Anything else on the GPU shows up here (Ollama unloads Llama after each call, so expect none).
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv
os.environ.update(VEIL_SAM_LOCAL="1", VEIL_SAM_PRECISION=PRECISION, VEIL_AUDITOR="none")
!python -m app.segmenter.sam3_core ../demo_assets/clips/manchester_street.mp4 --concept face --frames 240 --precision {PRECISION}
# Memory should level off after ~frame 60 (older tracker state is dropped as it goes).
# Each run segments 2-5 concepts over the whole clip, then re-checks the output once or twice.
# Rough batch time ≈ (frames per clip / the frames/s above) × 6 per run.

### Your own clip (optional)
Upload your recording (the desk shot from `demo_assets/props/RECORDING.md`) to `/content/` with the file browser, set `MY_CLIP` below, and run the cell. It's normalized to 1280 px / 30 fps and gets a sidecar with the instructions to run. If you already committed it with a sidecar, skip this.

In [ ]:
MY_CLIP = None  # e.g. "/content/desk_shot.mov"
if MY_CLIP:
    !python -m app.cache add-clip "{MY_CLIP}" --name desk_mine --title "My desk" \
        --prompt "Black out the laptop screen and the phone screen, and blur every face." \
        --follow-up "Also blur the ID badge."

In [ ]:
# 7 · Batch: run, freeze, render comparisons. Runs in a fresh process with SAM 3.1 loaded once.
# Runs that used a mock (masks, audit, or the rule-based planner standing in for Llama) are never
# cached: the build checks its models before starting and stops at the first run it would refuse.
# (Add --allow-mock-planner to keep runs whose plan came from the rule-based fallback.)
# Set here too, not just in step 6, so a restarted runtime can't fall back to the mocks.
os.environ.update(VEIL_SAM_LOCAL="1", VEIL_SAM_PRECISION=PRECISION, VEIL_AUDITOR="none")
args = f"--max-prompts {MAX_PROMPTS}" + ("" if FOLLOW_UPS else " --no-follow-ups")
if CLIPS:
    args += " --clips " + ",".join(CLIPS)
print("clips:", CLIPS or "all", "· follow-ups:", FOLLOW_UPS, "· planner:", os.environ.get("PLANNER_MODEL", "rule-based"))
!python -m app.cache build {args}
if _exit_code:  # set by the ! line above
    print(f"
build exited with code {_exit_code}" + (": killed, most likely out of RAM" if _exit_code in (137, -9) else ""))
    !free -h
!python -m app.cache list

In [ ]:
# 8 · Preview the first frozen run
import json, base64
from IPython.display import HTML, display
manifest = json.load(open("../demo_assets/cache/manifest.json"))
first = manifest[0]
print(first["instruction"], "·", first["models"])
video = base64.b64encode(open(f"../demo_assets/cache/{first['key']}/comparison_pair.mp4", "rb").read()).decode()
display(HTML(f'<video controls width="100%" src="data:video/mp4;base64,{video}"></video>'))

In [ ]:
# 9 · Take the cache home: unzip at the repo root (it holds demo_assets/...) and commit.
# Includes the clip you added in this session, so its replays have their "before" video.
!cd .. && zip -qr /content/veil_cache.zip demo_assets/cache $(ls demo_assets/clips/desk_mine.* 2>/dev/null) && unzip -l /content/veil_cache.zip | tail -1
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/veil_cache && cp /content/veil_cache.zip /content/drive/MyDrive/veil_cache/
from google.colab import files
files.download("/content/veil_cache.zip")

In [ ]:
# 10 · Optional: planner evaluation (~5 min). 14 instructions with known correct plans, scored for
# validity and correctness (right effect on each target, exceptions subtracted, nothing invented).
# Uses the planner from step 5. Don't run it while step 7 is building: both need the GPU.
import re
name = re.sub(r"[^a-z0-9]+", "_", os.environ.get("PLANNER_MODEL", "rule_based").lower()).strip("_")
!python -m app.planner.eval --repeat 2 --out /content/planner_eval_{name}.json
files.download(f"/content/planner_eval_{name}.json")